# 이전 문장 시제와 현재 문장 시제의 전환률 및 z점수 계산

- 2.개수_생성함수_sentence_tense.ipynb 파일에서 생성한 개수 생성 csv파일을 바탕으로 작성.


### log 
- ~2026.05.09
    - 입력: "..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_20260520_16-36.csv"
    - 출력: "..\csv\결과값\tense\RUNS_A_by_docu_id_2026-05-05_17-57.csv"(document별 RUNS 전환률 계산) 등
    - 필터 적용: df_1101_V = apply_filters(df, {"f_EN_No": 1101, "f_EN_label": "EF", "dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 20, "VX0_No": -1})


In [1]:
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH = Path(r"..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_20260520_16-36.csv")
        #FILTER_BODY_SEN_ENDS_NOT_QUOTE = lambda df: df[(df['sent_end_V'] == True) & (df['sent_end_V_in_quote'] == False) & (df['speaker'] == 'body')]

COUNT_MODE = "weight" #가중치 파일
WEIGHT_COL="count" #입력 파일이 이미 가중치가 적용된 형태이므로, 가중치 컬럼을 지정하여 "개수" 대신 "가중치 합계"로 계산하도록 함.

df = pd.read_csv(CSV_PATH)

print(f"CSV 파일 로드 완료: {len(df):,}, now: {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

#document 파일 읽기, 'docu_었_결합_등급', 'docu_었_결합_성향' 컬럼만 추출하여 df와 병합
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보_ver.1.2.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)
print(f"파일 읽기 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(f"df_docu: {df_docu.columns.tolist()}")
df_docu = df_docu[['docu_id', 'category', 'docu_었_결합_등급', 'docu_었_결합_성향']]

print(f"df_docu: {df_docu.columns.tolist()}")
df = df.merge(df_docu, on='docu_id', how='left')
print(f"df_docu와 병합 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

df.loc[df["문서범주"] == "책", "문서범주"] = "인문자연"

df['total'] = True

df.columns

CSV 파일 로드 완료: 1,022,399, now: 2026.06.18_06:52:53
파일 읽기 완료 - 2026.06.18_06:52:53
df_docu: ['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'docu_sen_count', 'docu_sen_count_has_E', 'docu_sen_count_not_quote', 'docu_sen_count_has_E_not_quote', 'docu_base_count_not_quote', 'docu_dominant_EN_No', 'docu_dominant_EN_label', 'docu_dominant_count', 'docu_dominant_ratio', 'docu_sent_count', 'docu_head_count', 'docu_body_count', 'docu_body_has_E_count', 'docu_body_not_quote_count', 'docu_body_not_quote_and_었_count', 'docu_었_결합_오즈비', 'docu_었_결합_로그오즈비', 'docu_었_결합_등급', 'docu_었_결합_성향', 'category2', 'file_sent_count', 'file_head_count', 'file_body_count', 'file_body_has_E_count', 'file_body_not_quote_count', 'file_body_not_quote_and_었_count', 'file_었_결합_오즈비', 'file_었_결합_로그오즈비', 'file_었_결합_등급', 'file_었_결합_성향']
df_docu: ['docu_id', 'category', 'docu_었_결합_등급', 'docu_었_결합_성향']
df_docu와 병합 완료 - 2026.06.

Index(['Unnamed: 0', '문서범주', 'category_x', '매체', 'file_id', 'docu_id',
       'speaker', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'V_No',
       'V_form', 'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No',
       'EN_No_sub', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No',
       'VX0_form', 'VX0_order', 'V0_form', 'V0_No', 'V0_label', 'f_EP_form',
       'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label',
       'has_prev_sentence', 'has_next_sentence', 'sentence_f_EP_form',
       'prev_sentence_f_EP_form', 'next_sentence_f_EP_form',
       'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote', 'EP_TT', 'EP_T', 'EP_M', 'f_EP_TT',
       'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T',

In [2]:
# 컬럼명 중복 제거(_x, _y)
y_cols = [c for c in df.columns if c.endswith("_y")]
df = df.drop(columns=y_cols)

df.columns = [c[:-2] if c.endswith("_x") else c for c in df.columns]

In [3]:
df.loc[df["문서범주"] == "책", "문서범주"] = "인문자연"
df["문서범주"].unique()

array(['신문', '인문자연', '체험', '허구'], dtype=object)

In [4]:
# 문서범주별로 전환률과 기대값 계산, 2연쇄 분석
# =========================================================
# Faster runs distribution + weighted transition summarizer
# =========================================================
import math
from functools import lru_cache
from typing import Any, Optional, Tuple

import numpy as np
import pandas as pd

# runs distribution 계산을 위한 helper 함수들
# - _normalize_binary_state: 다양한 형태의 T/F 값을 bool 또는 None으로 정규화
# - _two_sided_p_from_z: z-score로부터 양측 p-value 계산
# - _runs_pmf_cached: n_t, n_f에 대한 runs PMF를 계산하고 캐싱
# - runs_exact_p_values: 관측된 runs 수에 대한 정확한 p-value 계산

# 다양한 형태의 T/F 값을 bool 또는 None으로 정규화
def _normalize_binary_state(x: Any) -> Optional[bool]:
    if pd.isna(x):
        return None

    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    if isinstance(x, (int, np.integer, float, np.floating)):
        if x == 1:
            return True
        if x == 0:
            return False

    s = str(x).strip().upper()
    true_set = {"1", "T", "TRUE", "Y", "YES"}
    false_set = {"0", "F", "FALSE", "N", "NO"}

    if s in true_set:
        return True
    if s in false_set:
        return False

    raise ValueError(f"Could not normalize binary state: {x!r}")

# z-score로부터 양측 p-value 계산
def _two_sided_p_from_z(z: float) -> float:
    if pd.isna(z):
        return np.nan
    return math.erfc(abs(z) / math.sqrt(2))


@lru_cache(maxsize=None) # n_t, n_f에 대한 runs PMF를 캐싱하여 반복 계산을 빠르게 함, 메모리 사용량은 입력 범위에 따라 달라질 수 있음
def _runs_pmf_cached(n_t: int, n_f: int) -> Tuple[np.ndarray, np.ndarray]:
    n_t = int(n_t)
    n_f = int(n_f)

    if n_t < 0 or n_f < 0:
        raise ValueError("n_t and n_f must be nonnegative.")

    if n_t + n_f == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    if n_t == 0 or n_f == 0:
        return np.array([0], dtype=int), np.array([1.0], dtype=float)

    total = math.comb(n_t + n_f, n_t)
    ks = []
    probs = []

    r_max = 2 * min(n_t, n_f) + (1 if n_t != n_f else 0)

    for r in range(2, r_max + 1):
        if r % 2 == 0:
            m = r // 2
            ways = 2 * math.comb(n_t - 1, m - 1) * math.comb(n_f - 1, m - 1)
        else:
            m = (r - 1) // 2
            ways = 0
            if m <= n_t - 1 and m - 1 <= n_f - 1:
                ways += math.comb(n_t - 1, m) * math.comb(n_f - 1, m - 1)
            if m - 1 <= n_t - 1 and m <= n_f - 1:
                ways += math.comb(n_t - 1, m - 1) * math.comb(n_f - 1, m)

        if ways > 0:
            ks.append(r - 1)
            probs.append(ways / total)

    ks_arr = np.asarray(ks, dtype=int)
    probs_arr = np.asarray(probs, dtype=float)

    if probs_arr.size:
        probs_arr = probs_arr / probs_arr.sum()

    return ks_arr, probs_arr

# 관측된 runs 수에 대한 정확한 p-value 계산
def runs_exact_p_values(k_obs: int, n_t: int, n_f: int):
    ks, probs = _runs_pmf_cached(int(n_t), int(n_f))
    if ks.size == 0:
        return (np.nan, np.nan, np.nan)

    e_k = 2 * n_t * n_f / (n_t + n_f)

    p_lower = probs[ks <= k_obs].sum()
    p_upper = probs[ks >= k_obs].sum()

    obs_dist = abs(k_obs - e_k)
    p_two = probs[np.abs(ks - e_k) >= obs_dist - 1e-12].sum()

    return float(p_lower), float(p_upper), float(min(1.0, p_two))

# -----------------------
# 2연쇄 분석 함수
# -----------------------
# Main function1: analyze_runs_from_weighted_df - 전체/범주/문서용
# weighted sentence-level df에서 run 관련 통계를 계산하는 함수
def analyze_runs_from_weighted_df(
    df: pd.DataFrame,
    *,
    unit_col: str = "docu_id",
    state_col: str = "sentence_f_EP_T",
    prev_state_col: str = "prev_sentence_f_EP_T",
    has_prev_col: str = "has_prev_sentence",
    count_col: str = "count",
    add_exact_p: bool = True,
) -> pd.DataFrame:
    """
    weighted sentence-level df에서 run 관련 통계를 계산한다.

    전제
    ----
    - 각 row는 '현재 문장 상태' 1개를 대표한다.
    - count_col은 그런 row가 몇 개 있었는지를 뜻한다.
    - state_col은 현재 문장 T/F
    - prev_state_col은 이전 문장 T/F
    - has_prev_col은 실제 이전 문장 존재 여부
    """

    work = df.copy()

    work["_curr"] = work[state_col].map(_normalize_binary_state)
    work["_prev"] = work[prev_state_col].map(_normalize_binary_state)
    work["_has_prev"] = work[has_prev_col].map(_normalize_binary_state)

    # has_prev가 False면 prev값은 계산에서 쓰면 안 됨
    work.loc[work["_has_prev"] != True, "_prev"] = None

    # --------------------------
    # 1. 전체 상태 수: 모든 문장 기준
    # --------------------------
    curr_counts = (
        work.groupby([unit_col, "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    curr_pivot = curr_counts.pivot_table(
        index=unit_col,
        columns="_curr",
        values=count_col,
        fill_value=0,
    )
    # 
    all_units = pd.Index(sorted(work[unit_col].dropna().unique()), name=unit_col)
    out = pd.DataFrame(index=all_units)

    out["n_T"] = curr_pivot[True] if True in curr_pivot.columns else 0.0
    out["n_F"] = curr_pivot[False] if False in curr_pivot.columns else 0.0
    out["n"] = out["n_T"] + out["n_F"]

    
    # --------------------------
    # 2. sequence 개수
    #    has_prev == False 인 현재문장은 각 sequence의 시작
    # --------------------------
    seq_counts = (
        work.loc[work["_has_prev"] == False]
        .groupby(unit_col, observed=True)[count_col]
        .sum()
    )

    out["n_sequences"] = seq_counts.reindex(out.index, fill_value=0).astype(float)


    # --------------------------
    # 3. 전이표: has_prev == True 인 pair만
    # --------------------------
    pair_work = work.loc[
        (work["_has_prev"] == True)
        & (work["_prev"].notna())
        & (work["_curr"].notna())
    ].copy()

    pair_counts = (
        pair_work.groupby([unit_col, "_prev", "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    pair_pivot = pair_counts.pivot_table(
        index=unit_col,
        columns=["_prev", "_curr"],
        values=count_col,
        fill_value=0,
    )

    def get_pair(prev_val, curr_val):
        if len(pair_pivot) == 0:
            return pd.Series(0.0, index=out.index)
        if (prev_val, curr_val) in pair_pivot.columns:
            return pair_pivot[(prev_val, curr_val)].reindex(out.index, fill_value=0).astype(float)
        return pd.Series(0.0, index=out.index)

    out["a_TT"] = get_pair(True, True)
    out["b_TF"] = get_pair(True, False)
    out["c_FT"] = get_pair(False, True)
    out["d_FF"] = get_pair(False, False)

    out["k"] = out["b_TF"] + out["c_FT"]
    out["n_pairs"] = out["a_TT"] + out["b_TF"] + out["c_FT"] + out["d_FF"]

    # 이론적으로는 n - n_sequences 와 같아야 함
    out["expected_n_pairs_from_sequences"] = out["n"] - out["n_sequences"]
    out["pair_count_matches_structure"] = (
        out["n_pairs"] == out["expected_n_pairs_from_sequences"]
    )

    # --------------------------
    # 4. 전이율
    # --------------------------
    out["transition_rate"] = out["k"] / out["n_pairs"].replace(0, np.nan) # 전체 pair 대비 전환이 일어난 pair의 비율
    out["TF_transition_rate"] = out["b_TF"] / (out["a_TT"] + out["b_TF"]).replace(0, np.nan)
    out["FT_transition_rate"] = out["c_FT"] / (out["c_FT"] + out["d_FF"]).replace(0, np.nan)

    # --------------------------
    # 5. runs 기대값 / 분산 / z
    #    기본 runs 공식은 한 sequence 기준
    # --------------------------
    n = out["n"].replace(0, np.nan)

    out["E_k"] = 2 * out["n_T"] * out["n_F"] / n # runs의 기대값
    out["expected_transition_rate"] = out["E_k"]/ out["n_pairs"].replace(0, np.nan) # 기대 전환율
    out["delta_transition_rate"] = out["transition_rate"] - out["expected_transition_rate"] # 실제 전환율과 기대 전환율의 차이
    out["ratio_transition_rate"] = out["transition_rate"]/ out["expected_transition_rate"].replace(0, np.nan) # 실제 전환율이 기대 전환율의 몇 배인지

    out["Var_k"] = (
        2 * out["n_T"] * out["n_F"] * (2 * out["n_T"] * out["n_F"] - n)
    ) / (n**2 * (n - 1))

    out.loc[out["n"] <= 1, "Var_k"] = np.nan
    out["z_k"] = (out["k"] - out["E_k"]) / np.sqrt(out["Var_k"]) # runs 수의 z-score
    out["p_two_sided_z"] = out["z_k"].map(_two_sided_p_from_z) # z-score로부터 양측 p-value 계산

    # --------------------------
    # 6. exact p-value
    #    한 unit이 한 sequence일 때만 엄밀함
    # --------------------------
    if add_exact_p:
        p_lower = []
        p_upper = []
        p_two = []

        for _, row in out.iterrows():
            if row["n_sequences"] != 1:
                p_lower.append(np.nan)
                p_upper.append(np.nan)
                p_two.append(np.nan)
                continue

            vals = runs_exact_p_values(
                int(row["k"]),
                int(row["n_T"]),
                int(row["n_F"]),
            )
            p_lower.append(vals[0])
            p_upper.append(vals[1])
            p_two.append(vals[2])

        out["p_lower_exact"] = p_lower
        out["p_upper_exact"] = p_upper
        out["p_two_exact"] = p_two

    # --------------------------
    # 7. 2x2 association
    # --------------------------
    N = out["n_pairs"].replace(0, np.nan)
    out["E_a"] = (out["a_TT"] + out["b_TF"]) * (out["a_TT"] + out["c_FT"]) / N
    out["pearson_a"] = (out["a_TT"] - out["E_a"]) / np.sqrt(out["E_a"])

    a = out["a_TT"] + 0.5
    b = out["b_TF"] + 0.5
    c = out["c_FT"] + 0.5
    d = out["d_FF"] + 0.5

    out["OR"] = (a * d) / (b * c)
    out["log_OR"] = np.log(out["OR"])

    return out.reset_index()

# Main function2: analyze_transitions_against_baseline : 동사/어미/보조용언용
# baseline_col 기준 기대 전환율과 unit_col별 실제 전환율을 비교하는 함수
def analyze_transitions_against_baseline(
    df: pd.DataFrame,
    *,
    baseline_col: str,
    unit_col: str,
    baseline_df=None,   # 추가: baseline_df를 인자로 받아서 재사용할 수 있도록 함
    state_col: str = "sentence_f_EP_T",
    prev_state_col: str = "prev_sentence_f_EP_T",
    has_prev_col: str = "has_prev_sentence",
    count_col: str = "count",
) -> pd.DataFrame:
    """
    동사/어미/보조용언 등 특정 항목별 전이율을,
    baseline_col 기준 기대 전환율과 비교한다.

    예
    ----
    baseline_col = "문서범주"
    unit_col = "V_form"

    의미
    ----
    - 기대 전환율은 문서범주별 전체 구조에서 계산
    - 실제 전환율은 문서범주 × V_form별로 계산
    - 전환의확률비 = 실제 전환율 / baseline 기대 전환율
    """

    work = df.copy()

    work["_curr"] = work[state_col].map(_normalize_binary_state)
    work["_prev"] = work[prev_state_col].map(_normalize_binary_state)
    work["_has_prev"] = work[has_prev_col].map(_normalize_binary_state)

    # has_prev가 False이면 prev는 전이 계산에서 제외
    work.loc[work["_has_prev"] != True, "_prev"] = None

    # -------------------------------------------------
    # 1. baseline 계산
    #    기대 전환율은 baseline_col 기준으로 고정
    # -------------------------------------------------
    if baseline_df is None:
        baseline = analyze_runs_from_weighted_df(
            work,
            unit_col=baseline_col,
            state_col=state_col,
            prev_state_col=prev_state_col,
            has_prev_col=has_prev_col,
            count_col=count_col,
            add_exact_p=False,
        )
    else:
        baseline = baseline_df.copy()

    baseline_cols = [
        baseline_col,
        "n_T",
        "n_F",
        "n",
        "n_sequences",
        "n_pairs",
        "transition_rate",
        "expected_transition_rate",
        "E_k",
        "Var_k",
        "z_k",
        "OR",
        "log_OR",
    ]

    baseline = baseline[baseline_cols].rename(
        columns={
            "n_T": "baseline_n_T",
            "n_F": "baseline_n_F",
            "n": "baseline_n",
            "n_sequences": "baseline_n_sequences",
            "n_pairs": "baseline_n_pairs",
            "transition_rate": "baseline_transition_rate",
            "expected_transition_rate": "baseline_expected_transition_rate",
            "E_k": "baseline_E_k",
            "Var_k": "baseline_Var_k",
            "z_k": "baseline_z_k",
            "OR": "baseline_OR",
            "log_OR": "baseline_log_OR",
        }
    )

    # -------------------------------------------------
    # 2. unit별 실제 전이표 계산
    #    여기서는 E_k, Var_k를 새로 만들지 않음
    # -------------------------------------------------
    pair_work = work.loc[
        (work["_has_prev"] == True)
        & (work["_prev"].notna())
        & (work["_curr"].notna())
        & (work[unit_col].notna())
        & (work[baseline_col].notna())
    ].copy()

    g = (
        pair_work
        .groupby([baseline_col, unit_col, "_prev", "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    pivot = g.pivot_table(
        index=[baseline_col, unit_col],
        columns=["_prev", "_curr"],
        values=count_col,
        fill_value=0,
    )

    out = pd.DataFrame(index=pivot.index)

    def get_pair(prev_val, curr_val):
        if (prev_val, curr_val) in pivot.columns:
            return pivot[(prev_val, curr_val)].astype(float)
        return pd.Series(0.0, index=pivot.index)

    out["a_TT"] = get_pair(True, True)
    out["b_TF"] = get_pair(True, False)
    out["c_FT"] = get_pair(False, True)
    out["d_FF"] = get_pair(False, False)

    out["k"] = out["b_TF"] + out["c_FT"]
    out["n_pairs"] = out["a_TT"] + out["b_TF"] + out["c_FT"] + out["d_FF"]

    out["transition_rate"] = out["k"] / out["n_pairs"].replace(0, np.nan)
    out["TF_transition_rate"] = out["b_TF"] / (out["a_TT"] + out["b_TF"]).replace(0, np.nan)
    out["FT_transition_rate"] = out["c_FT"] / (out["c_FT"] + out["d_FF"]).replace(0, np.nan)

    # unit 자체의 T/F 분포도 참고용으로 계산
    curr_counts = (
        work.loc[
            work[unit_col].notna()
            & work[baseline_col].notna()
            & work["_curr"].notna()
        ]
        .groupby([baseline_col, unit_col, "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    curr_pivot = curr_counts.pivot_table(
        index=[baseline_col, unit_col],
        columns="_curr",
        values=count_col,
        fill_value=0,
    )

    out["unit_n_T"] = curr_pivot[True].reindex(out.index, fill_value=0).astype(float) if True in curr_pivot.columns else 0.0
    out["unit_n_F"] = curr_pivot[False].reindex(out.index, fill_value=0).astype(float) if False in curr_pivot.columns else 0.0
    out["unit_n"] = out["unit_n_T"] + out["unit_n_F"]
    out["unit_T_ratio"] = out["unit_n_T"] / out["unit_n"].replace(0, np.nan)

    out = out.reset_index()

    # -------------------------------------------------
    # 3. baseline 기대값 붙이기
    # -------------------------------------------------
    out = out.merge(
        baseline,
        on=baseline_col,
        how="left",
    )

    # -------------------------------------------------
    # 4. baseline 기준 비교 지표
    # -------------------------------------------------
    out["expected_transition_rate"] = out["baseline_expected_transition_rate"]

    out["delta_transition_rate"] = out["transition_rate"] - out["expected_transition_rate"] # 실제 전환율과 기대 전환율의 차이
    out["ratio_transition_OE"] = (out["transition_rate"] / out["expected_transition_rate"].replace(0, np.nan)) # 실제 전환율이 기대 전환율의 몇 배인지)

    out["transition_suppression_rate"] = 1 - out["ratio_transition_OE"] # 1보다 크면 오히려 기대보다 전환이 더 많이 일어난 것, 1보다 작으면 기대보다 전환이 억제된 것

    out["persistence_rate"] = 1 - out["transition_rate"]
    out["expected_persistence_rate"] = 1 - out["expected_transition_rate"]

    out["persistence_OE_ratio"] = (
        out["persistence_rate"] / out["expected_persistence_rate"].replace(0, np.nan)
    )

    # -------------------------------------------------
    # 5. unit 내부 2x2 연관성
    #    이건 기대전환률과 별개로 참고 가능
    # -------------------------------------------------
    a = out["a_TT"] + 0.5
    b = out["b_TF"] + 0.5
    c = out["c_FT"] + 0.5
    d = out["d_FF"] + 0.5

    out["unit_OR"] = (a * d) / (b * c)
    out["unit_log_OR"] = np.log(out["unit_OR"])

    return out


In [5]:
## 3연쇄 분석 1.
#-----------------------
# 3연쇄 분석함수
#-----------------------
# Main function3: analyze_trigram_from_weighted_df : 3연쇄 분석 함수, 전체/범주/문서용
def analyze_trigram_from_weighted_df(
    df,
    *,
    unit_col="docu_id",
    state_col="sentence_f_EP_T",
    prev_state_col="prev_sentence_f_EP_T",
    next_state_col="next_sentence_f_EP_T",
    has_prev_col="has_prev_sentence",
    has_next_col="has_next_sentence",
    count_col="count",
):
    work = df.copy()

    work["_prev"] = work[prev_state_col].map(_normalize_binary_state)
    work["_curr"] = work[state_col].map(_normalize_binary_state)
    work["_next"] = work[next_state_col].map(_normalize_binary_state)
    work["_has_prev"] = work[has_prev_col].map(_normalize_binary_state)
    work["_has_next"] = work[has_next_col].map(_normalize_binary_state)

    # -------------------------------------------------
    # 1. 전체 현재문장 기준 T/F 비율
    # -------------------------------------------------
    all_units = pd.Index(sorted(work[unit_col].dropna().unique()), name=unit_col)
    out = pd.DataFrame(index=all_units)

    curr_counts = (
        work.loc[
            work[unit_col].notna()
            & work["_curr"].notna()
        ]
        .groupby([unit_col, "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    curr_pivot = curr_counts.pivot_table(
        index=unit_col,
        columns="_curr",
        values=count_col,
        fill_value=0,
    )

    out["n_T"] = (
        curr_pivot[True].reindex(out.index, fill_value=0).astype(float)
        if True in curr_pivot.columns else 0.0
    )
    out["n_F"] = (
        curr_pivot[False].reindex(out.index, fill_value=0).astype(float)
        if False in curr_pivot.columns else 0.0
    )

    out["n_sentences"] = out["n_T"] + out["n_F"]
    out["P_T"] = out["n_T"] / out["n_sentences"].replace(0, np.nan)
    out["P_F"] = out["n_F"] / out["n_sentences"].replace(0, np.nan)

    # -------------------------------------------------
    # 2. 2연쇄 prev → curr 계산
    # -------------------------------------------------
    pair_work = work.loc[
        (work["_has_prev"] == True)
        & work["_prev"].notna()
        & work["_curr"].notna()
        & work[unit_col].notna()
    ].copy()

    pair_g = (
        pair_work
        .groupby([unit_col, "_prev", "_curr"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    pair_pivot = pair_g.pivot_table(
        index=unit_col,
        columns=["_prev", "_curr"],
        values=count_col,
        fill_value=0,
    )

    def get_pair(prev, curr):
        key = (prev, curr)
        if key in pair_pivot.columns:
            return pair_pivot[key].reindex(out.index, fill_value=0).astype(float)
        return pd.Series(0.0, index=out.index)

    out["TT"] = get_pair(True, True)
    out["TF"] = get_pair(True, False)
    out["FT"] = get_pair(False, True)
    out["FF"] = get_pair(False, False)

    pair_cols = ["TT", "TF", "FT", "FF"]

    out["n_pairs"] = out[pair_cols].sum(axis=1)

    out["P_T_given_T_pair"] = (out["TT"] / (out["TT"] + out["TF"]).replace(0, np.nan))
    out["P_F_given_T_pair"] = (out["TF"] / (out["TT"] + out["TF"]).replace(0, np.nan))
    out["P_T_given_F_pair"] = (out["FT"] / (out["FT"] + out["FF"]).replace(0, np.nan))
    out["P_F_given_F_pair"] = (out["FF"] / (out["FT"] + out["FF"]).replace(0, np.nan))

    # -------------------------------------------------
    # 3. 3연쇄 prev → curr → next 계산
    # -------------------------------------------------
    tri_work = work.loc[
        (work["_has_prev"] == True)
        & (work["_has_next"] == True)
        & work["_prev"].notna()
        & work["_curr"].notna()
        & work["_next"].notna()
        & work[unit_col].notna()
    ].copy()

    tri_g = (
        tri_work
        .groupby([unit_col, "_prev", "_curr", "_next"], observed=True)[count_col]
        .sum()
        .reset_index()
    )

    tri_pivot = tri_g.pivot_table(
        index=unit_col,
        columns=["_prev", "_curr", "_next"],
        values=count_col,
        fill_value=0,
    )

    def get_tri(prev, curr, next_):
        key = (prev, curr, next_)
        if key in tri_pivot.columns:
            return tri_pivot[key].reindex(out.index, fill_value=0).astype(float)
        return pd.Series(0.0, index=out.index)

    out["TTT"] = get_tri(True, True, True)
    out["TTF"] = get_tri(True, True, False)
    out["TFT"] = get_tri(True, False, True)
    out["TFF"] = get_tri(True, False, False)
    out["FTT"] = get_tri(False, True, True)
    out["FTF"] = get_tri(False, True, False)
    out["FFT"] = get_tri(False, False, True)
    out["FFF"] = get_tri(False, False, False)

    trigram_cols = ["TTT", "TTF", "TFT", "TFF", "FTT", "FTF", "FFT", "FFF"]
    out["n_trigrams"] = out[trigram_cols].sum(axis=1)

    # -------------------------------------------------
    # 4. 실제 3연쇄 조건부 확률
    # -------------------------------------------------
    for col in pair_cols: # TT, TF, FT, FF 각각에 대해 다음이 T/F일 확률 계산
        out[f"P_T_given_{col}"] = out[f"{col}T"] / (out[f"{col}T"] + out[f"{col}F"]).replace(0, np.nan)
        out[f"P_F_given_{col}"] = out[f"{col}F"] / (out[f"{col}T"] + out[f"{col}F"]).replace(0, np.nan)

    # -------------------------------------------------
    # 5. 1차 Markov 기대값
    # 공식: E(ABC) = count(AB) * P(C | B)
    # -------------------------------------------------
    out["E_TTT_markov"] = (out["TTT"] + out["TTF"]) * out["P_T_given_T_pair"]
    out["E_TTF_markov"] = (out["TTT"] + out["TTF"]) * out["P_F_given_T_pair"]

    out["E_TFT_markov"] = (out["TFT"] + out["TFF"]) * out["P_T_given_F_pair"]
    out["E_TFF_markov"] = (out["TFT"] + out["TFF"]) * out["P_F_given_F_pair"]

    out["E_FTT_markov"] = (out["FTT"] + out["FTF"]) * out["P_T_given_T_pair"]
    out["E_FTF_markov"] = (out["FTT"] + out["FTF"]) * out["P_F_given_T_pair"]

    out["E_FFT_markov"] = (out["FFT"] + out["FFF"]) * out["P_T_given_F_pair"]
    out["E_FFF_markov"] = (out["FFT"] + out["FFF"]) * out["P_F_given_F_pair"]

    # 기대확률
    for col in trigram_cols: #
        out[f"Obs_{col}_rate"] = out[col] / out["n_trigrams"].replace(0, np.nan)
        out[f"E_{col}_markov_rate"] = (
            out[f"E_{col}_markov"] / out["n_trigrams"].replace(0, np.nan)
        )

        # 실제 - 기대값
        out[f"{col}_markov_delta"] = (
            out[f"Obs_{col}_rate"] - out[f"E_{col}_markov_rate"]
        )

        # 실제 / 기대값
        out[f"{col}_markov_ratio"] = (
            out[col] / out[f"E_{col}_markov"].replace(0, np.nan)
        )

    # -------------------------------------------------
    # 6. 전체 T/F 비율 대비 비율차
    # -------------------------------------------------
    for col in pair_cols: # TT, TF, FT, FF 각각에 대해 다음 비율차 계산
        out[f"Diff_T_after_{col}"] = out[f"P_T_given_{col}"] - out["P_T"]
        out[f"Diff_F_after_{col}"] = out[f"P_F_given_{col}"] - out["P_F"]

    # -------------------------------------------------
    # 7. 전환 후 복귀/연장, 유지 후 유지/전환
    # -------------------------------------------------
    SET = [['TFT', 'FTF'],  ['TFF', 'FTT'], ['TTT', 'FFF'], ['TTF', 'FFT']]
    SET_NAMES = ['switch_return', 'switch_extension', 'stay_stay', 'stay_switch']

    for (col1, col2), name in zip(SET, SET_NAMES):
        out[f"{name}_Obs_rate"] = out[f"Obs_{col1}_rate"] + out[f"Obs_{col2}_rate"]
        out[f"{name}_E_rate"] = out[f"E_{col1}_markov_rate"] + out[f"E_{col2}_markov_rate"]
        out[f"{name}_delta_vs_baseline"] = out[f"{name}_Obs_rate"] - out[f"{name}_E_rate"]
        out[f"{name}_ratio_vs_baseline"] = out[f"{name}_Obs_rate"] / out[f"{name}_E_rate"].replace(0, np.nan)


    return out.reset_index()


In [13]:
#필터 적용 및 대략적인 데이터 분포 확인``
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))
from stats.filtering import apply_filters, FilterValue, has_value, _topn_values
df_1101 = apply_filters(df, {"f_EN_No": 1101, "f_EN_label": "EF"})
df_V = apply_filters(df, {"VX0_No": -1})
df_docu_selected_V = apply_filters(df, {"dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 20, "VX0_No": -1})
df_docu_selected_V = apply_filters(df, {"dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 20, "VX0_No": -1})

print(f"{len(df_1101):,} rows with f_EN_No=1101, label='EF'(다EF)),\n {len(df_V):,} rows with VX0_No=-1 (V),\n {len(df):,} total rows")
print(df["category"].value_counts(dropna=False))
print(df["prev_sentence_f_EP_T"].value_counts(dropna=False), df["has_prev_sentence"].value_counts(dropna=False))
print(f"{len(df)} rows with unique docu_id: {df['docu_id'].nunique()}")
print(f"{len(df_docu_selected_V)} rows with unique docu_id: {df_docu_selected_V['docu_id'].nunique()}")

834,511 rows with f_EN_No=1101, label='EF'(다EF)),
 716,183 rows with VX0_No=-1 (V),
 1,022,399 total rows
category
인문사회    314801
허구일반    289316
보도해설    171911
체험기술     88817
사설       47616
자연       32611
칼럼       27772
허구아동     27691
총류       21864
Name: count, dtype: int64
prev_sentence_f_EP_T
False    600701
True     421698
Name: count, dtype: int64 has_prev_sentence
True     980438
False     41961
Name: count, dtype: int64
1022399 rows with unique docu_id: 32721
500645 rows with unique docu_id: 9100


In [60]:
MOAD = "RUNS_A"
UNIT_COL = 'docu_id'# 'docu_id' #"file_id"#'total'
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")


out_df = analyze_runs_from_weighted_df(
    df_docu_selected_V,
    unit_col = UNIT_COL,
    state_col = "sentence_f_EP_T",
    prev_state_col = "prev_sentence_f_EP_T",
    has_prev_col = "has_prev_sentence",
    count_col = "count",
    add_exact_p = True,
)
print(f"analyze_runs_from_weighted_df 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

# =========================================================
# 4. save to file
# =========================================================
                                    
#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"

file_name = SAVE_DIR / f'{MOAD}_by_{UNIT_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
out_df.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"Output file for pivot table: {file_name}")


C:\Users\yu2hy\AppData\Local\Temp\ipykernel_28544\1430126694.py:143: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  work.loc[work["_has_prev"] != True, "_prev"] = None


analyze_runs_from_weighted_df 완료 - 2026.05.24_10:06:19
***저장합니다.:    2026-05-24 10:06:19.670792***
Output file for pivot table: ..\csv\결과값\tense\RUNS_A_by_docu_id_2026-05-24_10-06.csv


In [ ]:
MOAD = "RUNS_B"
BASE_COL = 'total'# 'docu_id' #"file_id"#'total'#'문서범주'
UNIT_COL = 'V_No' #'f_EN_No'
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")


# 1. 전체 baseline
baseline = analyze_runs_from_weighted_df(
    df_V,
    unit_col=BASE_COL
)

# 2. 일부 동사, 어미 등만 분석

df_1101_V = apply_filters(df, {"f_EN_No": 1101, "f_EN_label": "EF", "dominant_EN_No": 1101, "sen_count_has_E_not_quote": lambda s: s >= 20, "VX0_No": -1})

#target_verbs = ["하다", "되다", "가다"]
#df_sub = df_V #df_V[df_V["V_form"].isin(target_verbs)]

out_df = analyze_transitions_against_baseline(
    df_1101_V,
    baseline_col=BASE_COL,
    unit_col=UNIT_COL,
    baseline_df=baseline
)

print(f"analyze_transitions_against_baseline 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

# =========================================================
# 4. save to file
# =========================================================
                                    
#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"

file_name = SAVE_DIR / f'{MOAD}_{UNIT_COL}_by_{BASE_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
out_df.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"Output file for pivot table: {file_name}")


analyze_unit_trigram_against_baseline 완료 - 2026.05.28_22:42:01
***저장합니다.:    2026-05-28 22:42:01.110814***
Output file for pivot table: ..\csv\결과값\tense\RUNS_B_V_No_by_total_2026-05-28_22-41.csv


In [ ]:
#unit_effect unit별 3연쇄 분석 함수 -예전함수. 지금 버전은 4.4에 있음.

import numpy as np
import pandas as pd

#baseline_col별 현재 T/F 비율, 이전문장 T/F 비율, 다음문장 T/F 비율을 계산합니다. 첫 번째 파일의 baseline 함수
def analyze_chain_baseline_for_unit_effect(
    df: pd.DataFrame,
    *,
    baseline_col: str,
    state_col: str = "sentence_f_EP_T",
    prev_state_col: str = "prev_sentence_f_EP_T",
    next_state_col: str = "next_sentence_f_EP_T",
    has_prev_col: str = "has_prev_sentence",
    has_next_col: str = "has_next_sentence",
    count_col: str = "count",
) -> pd.DataFrame:
    """
    baseline_col별 기본 T/F 비율을 계산한다.

    반환값:
    - baseline_P_T / baseline_P_F
      : 현재문장 기준 전체 T/F 비율

    - baseline_prev_P_T / baseline_prev_P_F
      : 2연쇄 분석에서 사용할 이전문장 T/F 비율

    - baseline_next_P_T / baseline_next_P_F
      : 3연쇄 분석에서 사용할 다음문장 기본 T/F 비율
    """

    work = df.copy()

    if count_col not in work.columns:
        work[count_col] = 1

    work["_prev"] = work[prev_state_col].map(_normalize_binary_state)
    work["_curr"] = work[state_col].map(_normalize_binary_state)
    work["_next"] = work[next_state_col].map(_normalize_binary_state)
    work["_has_prev"] = work[has_prev_col].map(_normalize_binary_state)
    work["_has_next"] = work[has_next_col].map(_normalize_binary_state)

    # -------------------------------------------------
    # 1) 현재문장 기준 baseline P_T
    # -------------------------------------------------
    curr_work = work[
        work[baseline_col].notna()
        & work["_curr"].notna()
    ]

    curr_counts = (
        curr_work
        .groupby([baseline_col, "_curr"], observed=True)[count_col]
        .sum()
        .unstack("_curr", fill_value=0)
    )

    out = pd.DataFrame(index=curr_counts.index)

    out["baseline_n_T"] = curr_counts[True] if True in curr_counts.columns else 0.0
    out["baseline_n_F"] = curr_counts[False] if False in curr_counts.columns else 0.0
    out["baseline_n"] = out["baseline_n_T"] + out["baseline_n_F"]

    out["baseline_P_T"] = _rate(out["baseline_n_T"], out["baseline_n"])
    out["baseline_P_F"] = _rate(out["baseline_n_F"], out["baseline_n"])

    # -------------------------------------------------
    # 2) 2연쇄 분석용 prev T/F 비율
    # -------------------------------------------------
    prev_work = work[
        (work["_has_prev"] == True)
        & work[baseline_col].notna()
        & work["_prev"].notna()
        & work["_curr"].notna()
    ]

    prev_counts = (
        prev_work
        .groupby([baseline_col, "_prev"], observed=True)[count_col]
        .sum()
        .unstack("_prev", fill_value=0)
    )

    out = out.reindex(out.index.union(prev_counts.index))

    out["baseline_prev_n_T"] = (
        prev_counts[True].reindex(out.index, fill_value=0).astype(float)
        if True in prev_counts.columns else 0.0
    )
    out["baseline_prev_n_F"] = (
        prev_counts[False].reindex(out.index, fill_value=0).astype(float)
        if False in prev_counts.columns else 0.0
    )

    out["baseline_prev_n"] = out["baseline_prev_n_T"] + out["baseline_prev_n_F"]
    out["baseline_prev_P_T"] = _rate(out["baseline_prev_n_T"], out["baseline_prev_n"])
    out["baseline_prev_P_F"] = _rate(out["baseline_prev_n_F"], out["baseline_prev_n"])

    # -------------------------------------------------
    # 3) 3연쇄 분석용 next T/F 비율
    #    실제 3연쇄와 같은 조건: has_prev=True, has_next=True
    # -------------------------------------------------
    next_work = work[
        (work["_has_prev"] == True)
        & (work["_has_next"] == True)
        & work[baseline_col].notna()
        & work["_prev"].notna()
        & work["_curr"].notna()
        & work["_next"].notna()
    ]

    next_counts = (
        next_work
        .groupby([baseline_col, "_next"], observed=True)[count_col]
        .sum()
        .unstack("_next", fill_value=0)
    )

    out = out.reindex(out.index.union(next_counts.index))

    out["baseline_next_n_T"] = (
        next_counts[True].reindex(out.index, fill_value=0).astype(float)
        if True in next_counts.columns else 0.0
    )
    out["baseline_next_n_F"] = (
        next_counts[False].reindex(out.index, fill_value=0).astype(float)
        if False in next_counts.columns else 0.0
    )

    out["baseline_next_n"] = out["baseline_next_n_T"] + out["baseline_next_n_F"]
    out["baseline_next_P_T"] = _rate(out["baseline_next_n_T"], out["baseline_next_n"])
    out["baseline_next_P_F"] = _rate(out["baseline_next_n_F"], out["baseline_next_n"])

    return out.reset_index()

# Main function: analyze_unit_chain_effect : unit별 1·2·3연쇄를 계산하되, 2연쇄 기대값을 unit의 prev 분포 × baseline 현재 T/F 비율로 둡니다.
def analyze_unit_chain_effect(
    df: pd.DataFrame,
    *,
    baseline_col: str,
    unit_col: str,
    baseline_df: pd.DataFrame | None = None,
    state_col: str = "sentence_f_EP_T",
    prev_state_col: str = "prev_sentence_f_EP_T",
    next_state_col: str = "next_sentence_f_EP_T",
    has_prev_col: str = "has_prev_sentence",
    has_next_col: str = "has_next_sentence",
    count_col: str = "count",
    min_unit_n: int = 0,
    min_bigram_n: int = 0,
    min_trigram_n: int = 0,
) -> pd.DataFrame:
    """
    unit별 시제 연쇄 효과 분석.

    1연쇄:
    - unit_P_T vs baseline_P_T

    2연쇄:
    - 기대값 = unit_prev_P_T/F × baseline_P_T/F
    - 해당 unit이 놓인 이전문장 분포를 고정했을 때, 현재문장이 baseline T/F 비율대로 나오는가

    3연쇄:
    - 기대값 = 실제 Obs_TT/TF/FT/FF_rate × baseline_next_P_T/F
    - 실제 2연쇄 분포를 고정한 뒤 next가 기본 T/F 비율대로 가는지 봄.
    """

    work = df.copy()

    if count_col not in work.columns:
        work[count_col] = 1

    work["_prev"] = work[prev_state_col].map(_normalize_binary_state)
    work["_curr"] = work[state_col].map(_normalize_binary_state)
    work["_next"] = work[next_state_col].map(_normalize_binary_state)
    work["_has_prev"] = work[has_prev_col].map(_normalize_binary_state)
    work["_has_next"] = work[has_next_col].map(_normalize_binary_state)

    if baseline_df is None:
        baseline_df = analyze_chain_baseline_for_unit_effect(
            df,
            baseline_col=baseline_col,
            state_col=state_col,
            prev_state_col=prev_state_col,
            next_state_col=next_state_col,
            has_prev_col=has_prev_col,
            has_next_col=has_next_col,
            count_col=count_col,
        )

    # =====================================================
    # 1) unit별 현재문장 P_T
    # =====================================================
    curr_work = work[
        work[baseline_col].notna()
        & work[unit_col].notna()
        & work["_curr"].notna()
    ]

    curr_counts = (
        curr_work
        .groupby([baseline_col, unit_col, "_curr"], observed=True)[count_col]
        .sum()
        .unstack("_curr", fill_value=0)
    )

    out = pd.DataFrame(index=curr_counts.index)

    out["unit_n_T"] = curr_counts[True] if True in curr_counts.columns else 0.0
    out["unit_n_F"] = curr_counts[False] if False in curr_counts.columns else 0.0
    out["unit_n"] = out["unit_n_T"] + out["unit_n_F"]

    out["unit_P_T"] = _rate(out["unit_n_T"], out["unit_n"])
    out["unit_P_F"] = _rate(out["unit_n_F"], out["unit_n"])

    # =====================================================
    # 2) unit별 2연쇄 실제값
    # =====================================================
    bi_work = work[
        (work["_has_prev"] == True)
        & work[baseline_col].notna()
        & work[unit_col].notna()
        & work["_prev"].notna()
        & work["_curr"].notna()
    ]

    bi_counts = (
        bi_work
        .groupby([baseline_col, unit_col, "_prev", "_curr"], observed=True)[count_col]
        .sum()
        .unstack(["_prev", "_curr"], fill_value=0)
    )

    out = out.reindex(out.index.union(bi_counts.index))

    def get_bi(prev, curr):
        key = (prev, curr)
        if key in bi_counts.columns:
            return bi_counts[key].reindex(out.index, fill_value=0).astype(float)
        return pd.Series(0.0, index=out.index)

    out["TT"] = get_bi(True, True)
    out["TF"] = get_bi(True, False)
    out["FT"] = get_bi(False, True)
    out["FF"] = get_bi(False, False)

    bigram_cols = ["TT", "TF", "FT", "FF"]
    out["n_bigrams"] = out[bigram_cols].sum(axis=1)

    for col in bigram_cols:
        out[f"Obs_{col}_rate"] = _rate(out[col], out["n_bigrams"])

    # 조건부 2연쇄 확률
    out["P_curr_T_given_prev_T"] = _rate(out["TT"], out["TT"] + out["TF"])
    out["P_curr_F_given_prev_T"] = _rate(out["TF"], out["TT"] + out["TF"])

    out["P_curr_T_given_prev_F"] = _rate(out["FT"], out["FT"] + out["FF"])
    out["P_curr_F_given_prev_F"] = _rate(out["FF"], out["FT"] + out["FF"])

    # =====================================================
    # 3) unit별 3연쇄 실제값
    # =====================================================
    tri_work = work[
        (work["_has_prev"] == True)
        & (work["_has_next"] == True)
        & work[baseline_col].notna()
        & work[unit_col].notna()
        & work["_prev"].notna()
        & work["_curr"].notna()
        & work["_next"].notna()
    ]

    tri_counts = (
        tri_work
        .groupby([baseline_col, unit_col, "_prev", "_curr", "_next"], observed=True)[count_col]
        .sum()
        .unstack(["_prev", "_curr", "_next"], fill_value=0)
    )

    out = out.reindex(out.index.union(tri_counts.index))

    def get_tri(prev, curr, next_):
        key = (prev, curr, next_)
        if key in tri_counts.columns:
            return tri_counts[key].reindex(out.index, fill_value=0).astype(float)
        return pd.Series(0.0, index=out.index)

    tri_cols = ["TTT", "TTF", "TFT", "TFF", "FTT", "FTF", "FFT", "FFF"]

    out["TTT"] = get_tri(True, True, True)
    out["TTF"] = get_tri(True, True, False)

    out["TFT"] = get_tri(True, False, True)
    out["TFF"] = get_tri(True, False, False)

    out["FTT"] = get_tri(False, True, True)
    out["FTF"] = get_tri(False, True, False)

    out["FFT"] = get_tri(False, False, True)
    out["FFF"] = get_tri(False, False, False)

    out["n_trigrams"] = out[tri_cols].sum(axis=1)

    for col in tri_cols:
        out[f"Obs_{col}_rate"] = _rate(out[col], out["n_trigrams"])

    # 3연쇄에서 사용할 실제 2연쇄 분포
    # 주의: 2연쇄 전체가 아니라, 3연쇄 가능한 표본 안에서의 TT/TF/FT/FF 비율
    out["tri_TT"] = out["TTT"] + out["TTF"]
    out["tri_TF"] = out["TFT"] + out["TFF"]
    out["tri_FT"] = out["FTT"] + out["FTF"]
    out["tri_FF"] = out["FFT"] + out["FFF"]

    out["Obs_tri_TT_rate"] = _rate(out["tri_TT"], out["n_trigrams"])
    out["Obs_tri_TF_rate"] = _rate(out["tri_TF"], out["n_trigrams"])
    out["Obs_tri_FT_rate"] = _rate(out["tri_FT"], out["n_trigrams"])
    out["Obs_tri_FF_rate"] = _rate(out["tri_FF"], out["n_trigrams"])

    # 조건부 3연쇄 확률
    out["P_next_T_given_TT"] = _rate(out["TTT"], out["TTT"] + out["TTF"])
    out["P_next_F_given_TT"] = _rate(out["TTF"], out["TTT"] + out["TTF"])

    out["P_next_T_given_TF"] = _rate(out["TFT"], out["TFT"] + out["TFF"])
    out["P_next_F_given_TF"] = _rate(out["TFF"], out["TFT"] + out["TFF"])

    out["P_next_T_given_FT"] = _rate(out["FTT"], out["FTT"] + out["FTF"])
    out["P_next_F_given_FT"] = _rate(out["FTF"], out["FTT"] + out["FTF"])

    out["P_next_T_given_FF"] = _rate(out["FFT"], out["FFT"] + out["FFF"])
    out["P_next_F_given_FF"] = _rate(out["FFF"], out["FFT"] + out["FFF"])

    # =====================================================
    # 4) baseline 병합
    # =====================================================
    out = out.reset_index()

    out = out.merge(
        baseline_df,
        on=baseline_col,
        how="left",
    )

    # =====================================================
    # 5) 1연쇄 비교
    # =====================================================
    out["E_unit_P_T"] = out["baseline_P_T"]
    out["E_unit_P_F"] = out["baseline_P_F"]

    out["unit_P_T_delta_vs_baseline"] = out["unit_P_T"] - out["E_unit_P_T"]
    out["unit_P_F_delta_vs_baseline"] = out["unit_P_F"] - out["E_unit_P_F"]

    out["unit_P_T_ratio_vs_baseline"] = _ratio(out["unit_P_T"], out["E_unit_P_T"])
    out["unit_P_F_ratio_vs_baseline"] = _ratio(out["unit_P_F"], out["E_unit_P_F"])

    # =====================================================
    # 6) 2연쇄 기대값
    #
    # 기대값 =
    # unit의 실제 prev T/F 비율 × baseline의 현재 T/F 비율
    #
    # 의미:
    # 이 unit이 실제로 놓인 이전문장 T/F 환경은 인정한다.
    # 그 환경에서 현재문장이 baseline T/F 비율대로 나온다면
    # TT, TF, FT, FF가 어느 정도여야 하는지를 계산한다.
    # =====================================================

    # ---------------------------------
    # 6-1. unit의 실제 prev T/F 비율
    # ---------------------------------
    out["unit_prev_T_n"] = out["TT"] + out["TF"]
    out["unit_prev_F_n"] = out["FT"] + out["FF"]
    out["unit_prev_n"] = out["unit_prev_T_n"] + out["unit_prev_F_n"]

    out["unit_prev_P_T"] = _rate(out["unit_prev_T_n"], out["unit_prev_n"])
    out["unit_prev_P_F"] = _rate(out["unit_prev_F_n"], out["unit_prev_n"])

    # ---------------------------------
    # 6-2. 기대 2연쇄 비율
    # unit 실제 prev 분포 × baseline 현재 T/F 비율
    # ---------------------------------
    out["E_TT_rate"] = out["unit_prev_P_T"] * out["baseline_P_T"]
    out["E_TF_rate"] = out["unit_prev_P_T"] * out["baseline_P_F"]

    out["E_FT_rate"] = out["unit_prev_P_F"] * out["baseline_P_T"]
    out["E_FF_rate"] = out["unit_prev_P_F"] * out["baseline_P_F"]

    # 기대 빈도도 같이 만들어 두면 나중에 확인하기 좋음
    out["E_TT_n"] = out["n_bigrams"] * out["E_TT_rate"]
    out["E_TF_n"] = out["n_bigrams"] * out["E_TF_rate"]
    out["E_FT_n"] = out["n_bigrams"] * out["E_FT_rate"]
    out["E_FF_n"] = out["n_bigrams"] * out["E_FF_rate"]

    # ---------------------------------
    # 6-3. 실제값 - 기대값 / 실제값 ÷ 기대값
    # ---------------------------------
    for col in bigram_cols:
        out[f"{col}_delta_vs_expected"] = (
            out[f"Obs_{col}_rate"] - out[f"E_{col}_rate"]
        )

        out[f"{col}_ratio_vs_expected"] = _ratio(
            out[f"Obs_{col}_rate"],
            out[f"E_{col}_rate"]
        )

    # =====================================================
    # 6-4. 2연쇄 해석용 묶음
    # =====================================================

    # 유지: TT + FF
    out["stay_Obs_rate"] = out["Obs_TT_rate"] + out["Obs_FF_rate"]
    out["stay_E_rate"] = out["E_TT_rate"] + out["E_FF_rate"]
    out["stay_delta_vs_expected"] = (
        out["stay_Obs_rate"] - out["stay_E_rate"]
    )
    out["stay_ratio_vs_expected"] = _ratio(
        out["stay_Obs_rate"],
        out["stay_E_rate"]
    )

    # 전환: TF + FT
    out["switch_Obs_rate"] = out["Obs_TF_rate"] + out["Obs_FT_rate"]
    out["switch_E_rate"] = out["E_TF_rate"] + out["E_FT_rate"]
    out["switch_delta_vs_expected"] = (
        out["switch_Obs_rate"] - out["switch_E_rate"]
    )
    out["switch_ratio_vs_expected"] = _ratio(
        out["switch_Obs_rate"],
        out["switch_E_rate"]
    )

    # =====================================================
    # 6-5. prev가 curr에 미치는 방향별 영향
    #
    # 기준은 unit_P_T가 아니라 baseline_P_T
    # 즉, prev=T/F 조건에서 현재문장의 T 비율이
    # 기본 T 비율보다 얼마나 달라지는지를 본다.
    # =====================================================

    out["prev_T_to_curr_T_delta_vs_baseline_P_T"] = (
        out["P_curr_T_given_prev_T"] - out["baseline_P_T"]
    )

    out["prev_F_to_curr_T_delta_vs_baseline_P_T"] = (
        out["P_curr_T_given_prev_F"] - out["baseline_P_T"]
    )

    out["prev_T_to_curr_F_delta_vs_baseline_P_F"] = (
        out["P_curr_F_given_prev_T"] - out["baseline_P_F"]
    )

    out["prev_F_to_curr_F_delta_vs_baseline_P_F"] = (
        out["P_curr_F_given_prev_F"] - out["baseline_P_F"]
    )

    # prev=T일 때와 prev=F일 때 현재문장 T 비율이 얼마나 갈라지는가
    out["prev_chain_gap_on_curr_T"] = (
        out["P_curr_T_given_prev_T"] - out["P_curr_T_given_prev_F"]
    )

    # prev=T일 때와 prev=F일 때 현재문장 F 비율이 얼마나 갈라지는가
    out["prev_chain_gap_on_curr_F"] = (
        out["P_curr_F_given_prev_T"] - out["P_curr_F_given_prev_F"]
    )

    # =====================================================
    # 7) 3연쇄 기대값
    #
    # 기대값 =
    # 실제 TT/TF/FT/FF 비율 × baseline의 next T/F 비율
    # =====================================================

    out["E_TTT_rate"] = out["Obs_tri_TT_rate"] * out["baseline_next_P_T"]
    out["E_TTF_rate"] = out["Obs_tri_TT_rate"] * out["baseline_next_P_F"]

    out["E_TFT_rate"] = out["Obs_tri_TF_rate"] * out["baseline_next_P_T"]
    out["E_TFF_rate"] = out["Obs_tri_TF_rate"] * out["baseline_next_P_F"]

    out["E_FTT_rate"] = out["Obs_tri_FT_rate"] * out["baseline_next_P_T"]
    out["E_FTF_rate"] = out["Obs_tri_FT_rate"] * out["baseline_next_P_F"]

    out["E_FFT_rate"] = out["Obs_tri_FF_rate"] * out["baseline_next_P_T"]
    out["E_FFF_rate"] = out["Obs_tri_FF_rate"] * out["baseline_next_P_F"]

    for col in tri_cols:
        out[f"{col}_delta_vs_expected"] = out[f"Obs_{col}_rate"] - out[f"E_{col}_rate"]
        out[f"{col}_ratio_vs_expected"] = _ratio(out[f"Obs_{col}_rate"], out[f"E_{col}_rate"])

    # 3연쇄 조건부 비교
    # 각 TT/TF/FT/FF 뒤의 next=T/F가 baseline_next_P_T/F보다 얼마나 다른가
    out["next_T_after_TT_delta_vs_baseline"] = out["P_next_T_given_TT"] - out["baseline_next_P_T"]
    out["next_T_after_TF_delta_vs_baseline"] = out["P_next_T_given_TF"] - out["baseline_next_P_T"]
    out["next_T_after_FT_delta_vs_baseline"] = out["P_next_T_given_FT"] - out["baseline_next_P_T"]
    out["next_T_after_FF_delta_vs_baseline"] = out["P_next_T_given_FF"] - out["baseline_next_P_T"]

    # =====================================================
    # 8) 복귀/지속/유지/전환 묶음 지표
    # =====================================================

    # 전환 후 복귀: TFT, FTF
    out["switch_return_Obs_rate"] = out["Obs_TFT_rate"] + out["Obs_FTF_rate"]
    out["switch_return_E_rate"] = out["E_TFT_rate"] + out["E_FTF_rate"]
    out["switch_return_delta_vs_expected"] = (
        out["switch_return_Obs_rate"] - out["switch_return_E_rate"]
    )
    out["switch_return_ratio_vs_expected"] = _ratio(
        out["switch_return_Obs_rate"],
        out["switch_return_E_rate"]
    )

    # 전환 후 지속: TFF, FTT
    out["switch_extension_Obs_rate"] = out["Obs_TFF_rate"] + out["Obs_FTT_rate"]
    out["switch_extension_E_rate"] = out["E_TFF_rate"] + out["E_FTT_rate"]
    out["switch_extension_delta_vs_expected"] = (
        out["switch_extension_Obs_rate"] - out["switch_extension_E_rate"]
    )
    out["switch_extension_ratio_vs_expected"] = _ratio(
        out["switch_extension_Obs_rate"],
        out["switch_extension_E_rate"]
    )

    # 유지 후 유지: TTT, FFF
    out["stay_stay_Obs_rate"] = out["Obs_TTT_rate"] + out["Obs_FFF_rate"]
    out["stay_stay_E_rate"] = out["E_TTT_rate"] + out["E_FFF_rate"]
    out["stay_stay_delta_vs_expected"] = (
        out["stay_stay_Obs_rate"] - out["stay_stay_E_rate"]
    )
    out["stay_stay_ratio_vs_expected"] = _ratio(
        out["stay_stay_Obs_rate"],
        out["stay_stay_E_rate"]
    )

    # 유지 후 전환: TTF, FFT
    out["stay_switch_Obs_rate"] = out["Obs_TTF_rate"] + out["Obs_FFT_rate"]
    out["stay_switch_E_rate"] = out["E_TTF_rate"] + out["E_FFT_rate"]
    out["stay_switch_delta_vs_expected"] = (
        out["stay_switch_Obs_rate"] - out["stay_switch_E_rate"]
    )
    out["stay_switch_ratio_vs_expected"] = _ratio(
        out["stay_switch_Obs_rate"],
        out["stay_switch_E_rate"]
    )

    # 방향별 복귀율
    # TF 뒤에 T가 오면 원래 T로 복귀
    out["return_after_TF_Obs_rate"] = out["P_next_T_given_TF"]
    out["return_after_TF_E_rate"] = out["baseline_next_P_T"]
    out["return_after_TF_delta_vs_expected"] = (
        out["return_after_TF_Obs_rate"] - out["return_after_TF_E_rate"]
    )

    # FT 뒤에 F가 오면 원래 F로 복귀
    out["return_after_FT_Obs_rate"] = out["P_next_F_given_FT"]
    out["return_after_FT_E_rate"] = out["baseline_next_P_F"]
    out["return_after_FT_delta_vs_expected"] = (
        out["return_after_FT_Obs_rate"] - out["return_after_FT_E_rate"]
    )

    # 방향별 지속율
    out["extension_after_TF_Obs_rate"] = out["P_next_F_given_TF"]
    out["extension_after_TF_E_rate"] = out["baseline_next_P_F"]
    out["extension_after_TF_delta_vs_expected"] = (
        out["extension_after_TF_Obs_rate"] - out["extension_after_TF_E_rate"]
    )

    out["extension_after_FT_Obs_rate"] = out["P_next_T_given_FT"]
    out["extension_after_FT_E_rate"] = out["baseline_next_P_T"]
    out["extension_after_FT_delta_vs_expected"] = (
        out["extension_after_FT_Obs_rate"] - out["extension_after_FT_E_rate"]
    )

    # =====================================================
    # 9) 최소 빈도 필터
    # =====================================================
    if min_unit_n > 0:
        out = out[out["unit_n"] >= min_unit_n]

    if min_bigram_n > 0:
        out = out[out["n_bigrams"] >= min_bigram_n]

    if min_trigram_n > 0:
        out = out[out["n_trigrams"] >= min_trigram_n]

    return out

In [14]:
df.columns

Index(['Unnamed: 0', '문서범주', 'category', '매체', 'file_id', 'docu_id', 'speaker',
       'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'V_No',
       'V_form', 'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No',
       'EN_No_sub', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No',
       'VX0_form', 'VX0_order', 'V0_form', 'V0_No', 'V0_label', 'f_EP_form',
       'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label',
       'has_prev_sentence', 'has_next_sentence', 'sentence_f_EP_form',
       'prev_sentence_f_EP_form', 'next_sentence_f_EP_form',
       'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote', 'EP_TT', 'EP_T', 'EP_M', 'f_EP_TT',
       'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T',
 

In [71]:
MOAD = "TRI_A"
UNIT_COL = 'docu_id'#'문서범주'#'docu_id'# 'docu_id' #"file_id"#'total'
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

tri_df = analyze_trigram_from_weighted_df(
    df_docu_selected_V,
    unit_col=UNIT_COL,
)
print(f"analyze_runs_from_weighted_df 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

# =========================================================
# 4. save to file
# =========================================================
# category 병합
if 'docu_id' in tri_df.columns:
    docu_info = (df[['docu_id', '문서범주']].drop_duplicates())

    tri_df = tri_df.merge(docu_info, on='docu_id', how='left')
#---- file name settings ----  
SAVE_DIR = Path("..") / "csv"/"결과값"/"tense"
file_name = SAVE_DIR / f'{MOAD}_by_{UNIT_COL}_{timestamp}.csv'

print(f"***저장합니다.:    {datetime.now()}***")
tri_df.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"Output file for pivot table: {file_name}")

analyze_runs_from_weighted_df 완료 - 2026.05.28_21:46:30
***저장합니다.:    2026-05-28 21:46:30.951613***
Output file for pivot table: ..\csv\결과값\tense\TRI_A_by_docu_id_2026-05-28_21-46.csv


In [62]:
tri_df.head(5)

,docu_id,n_T,n_F,n_sentences,P_T,P_F,TT,TF,FT,FF,...,switch_extension_ratio_vs_baseline,stay_stay_Obs_rate,stay_stay_E_rate,stay_stay_delta_vs_baseline,stay_stay_ratio_vs_baseline,stay_switch_Obs_rate,stay_switch_E_rate,stay_switch_delta_vs_baseline,stay_switch_ratio_vs_baseline,문서범주
0,AA0001.021,1.0,20.0,21.0,0.047619,0.952381,0.0,1.0,1.0,18.0,...,1.055556,0.842105,0.847645,-0.005540,0.993464,0.052632,0.047091,0.005540,1.117647,신문
1,AA0001.065,1.0,18.0,19.0,0.052632,0.947368,0.0,1.0,1.0,16.0,...,1.062500,0.823529,0.830450,-0.006920,0.991667,0.058824,0.051903,0.006920,1.133333,신문
2,AA0001.071,6.0,18.0,24.0,0.250000,0.750000,5.0,1.0,0.0,17.0,...,1.000000,0.909091,0.916667,-0.007576,0.991736,0.045455,0.037879,0.007576,1.200000,신문
3,AA0002.232,1.0,22.0,23.0,0.043478,0.956522,0.0,1.0,1.0,20.0,...,1.050000,0.857143,0.861678,-0.004535,0.994737,0.047619,0.043084,0.004535,1.105263,신문
4,AA0003.001,6.0,25.0,31.0,0.193548,0.806452,4.0,2.0,2.0,22.0,...,1.263158,0.724138,0.755747,-0.031609,0.958175,0.137931,0.106322,0.031609,1.297297,신문


In [63]:
df.columns

Index(['Unnamed: 0', '문서범주', 'category', '매체', 'file_id', 'docu_id', 'speaker',
       'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'V_No',
       'V_form', 'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No',
       'EN_No_sub', 'VX_len', 'Next_VX_No', 'Next_VX_form', 'VX0_No',
       'VX0_form', 'VX0_order', 'V0_form', 'V0_No', 'V0_label', 'f_EP_form',
       'f_EN_form', 'f_EN_No', 'f_EN_No_sub', 'f_EN_label',
       'has_prev_sentence', 'has_next_sentence', 'sentence_f_EP_form',
       'prev_sentence_f_EP_form', 'next_sentence_f_EP_form',
       'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote', 'EP_TT', 'EP_T', 'EP_M', 'f_EP_TT',
       'f_EP_T', 'f_EP_M', 'sentence_f_EP_TT', 'sentence_f_EP_T',
       'sentence_f_EP_M', 'prev_sentence_f_EP_TT', 'prev_sentence_f_EP_T',
 